# 🔍 EDA Basics — Exploratory Data Analysis

> **What is EDA?**
>
> EDA means **"getting to know your data"** before doing any serious analysis or machine learning.
>
> Before a doctor treats a patient, they run tests to understand the situation. EDA is the same thing for data — you run "tests" (functions) to understand your dataset:
> - How big is it?
> - What columns does it have?
> - Are there missing values?
> - Are there weird outliers?
>
> **Skipping EDA = building on a foundation you don't understand.** Always do it first!
>
> ### Libraries we'll use:
> | Library | What it does |
> |---|---|
> | `pandas` | Data manipulation and analysis |
> | `seaborn` | Statistical data visualization (built on matplotlib) |
> | `numpy` | Numerical computing (arrays, math) |
> | `matplotlib` | Base plotting library |

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

print("✅ All libraries imported successfully!")
print(f"  pandas   version: {pd.__version__}")
print(f"  seaborn  version: {sns.__version__}")
print(f"  numpy    version: {np.__version__}")

---
## 📦 Loading Built-in Datasets

Seaborn comes with several **famous practice datasets** built-in. You don't need to download anything.

| Dataset | What it contains |
|---|---|
| `"titanic"` | Passenger data from the Titanic disaster (survived/not, age, class, etc.) |
| `"iris"` | Measurements of 3 species of flowers |
| `"tips"` | Restaurant tips data (bill amount, tip, time, day, etc.) |

We'll use **Titanic** — it's the classic beginner dataset.

In [ ]:
df = sns.load_dataset("titanic")

print(f"✅ Titanic dataset loaded!")
print(f"   Shape: {df.shape}  →  {df.shape[0]} passengers, {df.shape[1]} columns")
print()
print("First 5 rows:")
df.head()

---
## 🗂️ `df.info()` — The First Thing You Run

`df.info()` gives you the **X-ray** of your dataset in one shot:
- Number of rows and columns
- Column names and their **data types** (int, float, object/string, bool)
- How many **non-null** (non-missing) values each column has

### ⚠️ Common Mistake
```python
df.info    # ❌ Wrong! This just shows the method object, doesn't run it
df.info()  # ✅ Correct! The () actually calls/runs the function
```
Always use `()` to call a function!

In [ ]:
# df.info() — read this like a report card for your data
# Look for:
#   - Columns with fewer non-null values → those have MISSING DATA
#   - 'object' dtype → means text/string column
#   - 'float64' vs 'int64' → decimals vs whole numbers
df.info()

In [ ]:
# Let's highlight missing data clearly
print("=== Missing Values Report ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if len(missing_df) > 0:
    print(missing_df)
    print()
    print("⚠️  'deck' column is missing 77% of values — probably not useful")
    print("⚠️  'age' is missing ~20% — we'll need to handle this")
else:
    print("✅ No missing values!")

---
## 📊 `df.describe()` — Statistical Summary

`df.describe()` automatically calculates key statistics for every **numeric column**:

| Statistic | Meaning |
|---|---|
| `count` | How many non-missing values |
| `mean` | Average |
| `std` | Standard deviation (how spread out values are) |
| `min` | Minimum value |
| `25%` | 25th percentile (bottom quarter) |
| `50%` | Median (middle value) |
| `75%` | 75th percentile (top quarter) |
| `max` | Maximum value |

In [ ]:
df.describe()

In [ ]:
# Quick insights from describe()
print("=== Key Insights from describe() ===")
print(f"Average passenger age: {df['age'].mean():.1f} years")
print(f"Youngest passenger:    {df['age'].min()} years")
print(f"Oldest passenger:      {df['age'].max()} years")
print(f"Average fare paid:     ${df['fare'].mean():.2f}")
print(f"Most expensive ticket: ${df['fare'].max():.2f}")
print(f"Survival rate:         {df['survived'].mean()*100:.1f}%")

---
## 🧹 Handling Missing Values (Data Cleaning)

Real-world data is **messy**. Missing values (shown as `NaN` — Not a Number) are everywhere.

We'll use a separate, messier dataset for this part.

### 3 ways to handle missing values:
| Method | What it does | When to use |
|---|---|---|
| `dropna()` | **Delete** rows with any NaN | When you have lots of data and can afford to lose some rows |
| `fillna(value)` | **Replace** NaN with a fixed value | When you know what value makes sense |
| `fillna(mean/median)` | **Replace** NaN with average/median | Most common — fill with "typical" value |

In [ ]:
# Load a dataset with intentional missing values for practice
df_w3 = pd.read_csv("../Datasets/cleaning_dataset.csv")

print(f"Dataset shape: {df_w3.shape}")
print()
print("Missing values per column:")
print(df_w3.isnull().sum())
print()
df_w3.head(10)

In [ ]:
# METHOD 1: dropna() — remove rows that have ANY missing value
print(f"Rows before dropna(): {len(df_w3)}")

df_cleaned = df_w3.dropna()  # This creates a NEW DataFrame (original is safe)

print(f"Rows after dropna():  {len(df_cleaned)}")
print(f"Rows removed: {len(df_w3) - len(df_cleaned)}")
print()
print("⚠️  dropna() without inplace=True doesn't change df_w3 — it creates a copy")
print("   To modify the original: df_w3.dropna(inplace=True)")

In [ ]:
# METHOD 1b: inplace=True — modifies the DataFrame directly
# (reload the data first since we want to show other methods too)
df_w3 = pd.read_csv("../Datasets/cleaning_dataset.csv")

# Without inplace: creates a new DataFrame
# df_cleaned = df_w3.dropna()  

# With inplace=True: modifies df_w3 itself
df_w3.dropna(inplace=True)

print(f"df_w3 now has {len(df_w3)} rows (modified in-place)")

In [ ]:
# Reload clean data to demonstrate fillna()
df_w3 = pd.read_csv("../Datasets/cleaning_dataset.csv")

# METHOD 2: fillna() with median — fill missing Calories with the median calorie value
# WHY median and not mean? Because median isn't affected by outliers!
calories_median = df_w3["Calories"].median()
print(f"Median calories: {calories_median}")
df_w3["Calories"].fillna(calories_median, inplace=True)
print(f"Missing Calories after fillna: {df_w3['Calories'].isnull().sum()}  ← should be 0")
print()

# Fill Duration with MODE (most common value)
# WHY mode? Duration is likely a few specific values (30, 45, 60 min)
duration_mode = df_w3["Duration"].mode()[0]
print(f"Mode of Duration: {duration_mode}")
df_w3["Duration"].fillna(duration_mode, inplace=True)
print(f"Missing Duration after fillna: {df_w3['Duration'].isnull().sum()}  ← should be 0")
print()

# Fill Maxpulse with mode
df_w3.fillna({"Maxpulse": df_w3["Maxpulse"].mode()[0]}, inplace=True)
print(f"Missing Maxpulse after fillna: {df_w3['Maxpulse'].isnull().sum()}  ← should be 0")
print()
print("✅ All missing values handled!")
print()
print("Remaining missing values:")
print(df_w3.isnull().sum())